In [1]:
# 将前台中的需要单独拆分的业务线拉出来，单独计算
from mypackage.utilities import connect_to_db, business_line_split
from mypackage.mapping import reverse_combined_column_mapping
import pandas as pd
import os
import shutil
import numpy as np
from mypackage.config import bus_lines,get_date_range,groups_frontend,get_filename,get_foldername,get_test_date

# 设定日期范围
date_range=get_date_range()

folder_name=get_foldername()
bus_lines = bus_lines
conn,cur=connect_to_db()
# 获取全部层级
cur.execute(
  f"SELECT * FROM dim_org_struc WHERE unique_lvl not like '%无归属%'"
)
df=cur.fetchall()

columns = [reverse_combined_column_mapping.get(desc[0]) for desc in cur.description]
df=pd.DataFrame(df,columns=columns)
# 导出全部业务线层级
# df.to_csv('全部业务线层级.csv',index=False)
levels=df['唯一层级'].drop_duplicates().tolist()

# 获取全部收入
cur.execute(
  f"SELECT * FROM fact_revenue"
)
df=cur.fetchall()
columns = [reverse_combined_column_mapping.get(desc[0]) for desc in cur.description]
df_revenue=pd.DataFrame(df,columns=columns)
df_revenue=df_revenue[df_revenue['会计期间'].isin(date_range)]
df_revenue=df_revenue[df_revenue['唯一层级'].isin(levels)]
df_revenue=df_revenue.drop(['一级组织','二级组织','三级组织'],axis=1)
df_revenue[['一级组织','二级组织','三级组织']]=df_revenue['唯一层级'].str.split('-',n=2,expand=True)
df_revenue=df_revenue[['来源编号','一级组织','二级组织','三级组织','会计期间','收入大类','产品大类','物料名称','不含税金额本位币','成本金额','运费成本','关税成本','软件成本','唯一层级','年份']]
df_revenue[bus_lines] = np.nan
df_revenue.rename(columns={'不含税金额本位币':'金额'},inplace=True)


# 获取全部其他项目
cur.execute(
  f"SELECT * FROM fact_profit_bd"
)
df=cur.fetchall()
acct_list=["信用减值损失", "公允价值变动收益" , "其他收益" , "所得税费用" , "投资收益" , "税金及附加" , "营业外支出" , "营业外收入" , "资产减值损失" , "资产处置收益"]
columns = [reverse_combined_column_mapping.get(desc[0]) for desc in cur.description]
df_profit=pd.DataFrame(df,columns=columns)
df_profit=df_profit[df_profit['日期'].isin(date_range)]
df_profit=df_profit[df_profit['唯一层级'].isin(levels) & df_profit['一级科目'].isin(acct_list)]
df_profit['年份']=pd.to_datetime(df_profit['日期']).dt.year.astype(str)
df_profit=df_profit.drop(['一级组织','二级组织','三级组织'],axis=1)
df_profit[['一级组织','二级组织','三级组织']]=df_profit['唯一层级'].str.split('-',n=2,expand=True)
df_profit=df_profit[['来源编号','财报合并','一级组织','二级组织','三级组织','日期','一级科目','本月金额','唯一层级','年份']]
df_profit[bus_lines] = np.nan
# 去掉零值
# 步骤1：先移除空值
df_profit = df_profit.dropna(subset=['本月金额'])
# 步骤2：再过滤掉0值
df_profit = df_profit[df_profit['本月金额'] != 0]
print("验证是否有空值：",df_profit['本月金额'].isna().sum())  # 应输出0
print("验证是否有0值：",(df_profit['本月金额'] == 0).sum())  # 应输出0

df_profit.rename(columns={'本月金额':'金额'},inplace=True)

# Excel 文件路径和工作表名
sheet_name = '部门金额'
groups = groups_frontend

# 拆分收入数据

errored_files = []
try:
    cur.execute("SELECT distinct unique_lvl, prim_org, sec_org, short_name, category FROM dim_org_struc")
    df_path = pd.DataFrame(cur.fetchall(), columns=['unique_lvl', 'prim_org', 'sec_org', 'short_name', 'category'])

    for group in groups:
        try:
            unique_lvls = df_path[df_path.short_name == group].unique_lvl.to_list()
            print('本次用户组：', group, '本次层级包括：', unique_lvls)
            prim_org = df_path[df_path.short_name == group].prim_org.to_list()[0]
            sec_org = df_path[df_path.short_name == group].sec_org.to_list()[0]

            # 组合文件路径
            file_path = os.path.join('Z:\\', '11-业务报表', '8.业务线核算', prim_org, sec_org,folder_name, get_filename('业务线收入拆分表'))
            source_file_path = r'D:\新国都\4.业报\7.BI驾驶舱\业务线收入拆分表.xlsx'
            shutil.copy(source_file_path, file_path)
            print(file_path)
            
            business_line_split(df_revenue, file_path, sheet_name,unique_lvls,date_range,'会计期间')
            
            
            # 组合文件路径
            file_path = os.path.join('Z:\\', '11-业务报表', '8.业务线核算', prim_org, sec_org,folder_name, get_filename('业务线其他拆分表'))
            source_file_path = r'D:\新国都\4.业报\7.BI驾驶舱\业务线其他拆分表.xlsx'
            shutil.copy(source_file_path, file_path)
            print(file_path)
            
            business_line_split(df_profit, file_path, sheet_name, unique_lvls,date_range,'日期')
            
        except Exception as e:
            print(f"Error processing group {group}: {e}")
            errored_files.append(file_path)
finally:
    conn.close()
    if errored_files:
        print("Error encountered with the following files:")
        for file in errored_files:
            print(file)
               


验证是否有空值： 0
验证是否有0值： 0
本次用户组： dcpmc_center 本次层级包括： ['国内渠道事业群-合伙人营销中心-合伙人发展部', '国内渠道事业群-合伙人营销中心-公共部门', '国内渠道事业群-合伙人营销中心-项目支持部']
Z:\11-业务报表\8.业务线核算\国内渠道事业群\合伙人营销中心\2026年02月\业务线收入拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\国内渠道事业群\合伙人营销中心\2026年02月\业务线其他拆分表202602.xlsx
数据已成功写入 Excel 文件。
本次用户组： psacc_center 本次层级包括： ['支付服务事业群-审核能力中心-互联网业务中心', '支付服务事业群-审核能力中心-总经办', '支付服务事业群-审核能力中心-项目中心', '支付服务事业群-审核能力中心-金融业务中心', '支付服务事业群-审核能力中心-创新业务中心', '支付服务事业群-审核能力中心-公共部门', '支付服务事业群-审核能力中心-渠道管理中心', '支付服务事业群-审核能力中心-职能支持部', '支付服务事业群-审核能力中心-客户服务中心', '支付服务事业群-审核能力中心-运营中心', '支付服务事业群-审核能力中心-技术中心']
Z:\11-业务报表\8.业务线核算\支付服务事业群\审核能力中心\2026年02月\业务线收入拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\支付服务事业群\审核能力中心\2026年02月\业务线其他拆分表202602.xlsx
数据已成功写入 Excel 文件。
本次用户组： omc_center 本次层级包括： ['国内渠道事业群-运营中心-服务商管理部', '国内渠道事业群-运营中心-业务运营部', '国内渠道事业群-运营中心-公共部门', '国内渠道事业群-运营中心-业务管理部']
Z:\11-业务报表\8.业务线核算\国内渠道事业群\运营中心\2026年02月\业务线收入拆分表202602.xlsx
数据已成功写入 Excel 文件。
Z:\11-业务报表\8.业务线核算\国内渠道事业群\运营中心\2026年02月\业务线其他拆分表202602.xlsx
